# Modelagem exploratória de preços

O alvo é o preço anunciado. A amostra de demonstração é sintética. Execute o pipeline com dados adequados para estudar o desempenho de mercado. A divisão aleatória aqui é exploratória e não mede generalização temporal.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pathlib import Path
import sys

root = Path.cwd().resolve()
if not (root / "src" / "storage.py").is_file():
    root = root.parent
sys.path.insert(0, str(root))
from src.storage import CloudStorage

sns.set_theme(style="whitegrid")

Carga de Dados

In [ ]:
df = CloudStorage().ler_csv("carros_dados_limpos.csv").dropna(subset=["Preço", "Ano"])
if len(df) < 10:
    raise ValueError("São necessários pelo menos dez anúncios válidos para esta demonstração.")
display(df.head())

Engenharia de Features

In [ ]:
df["Motor"] = df["Descrição"].str.extract(r"\b(\d\.\d)\b", expand=False).astype(float)
ano_referencia = pd.Timestamp.now(tz="UTC").year
df["Idade"] = (ano_referencia - df["Ano"]).clip(lower=0)
df["Km_por_Ano"] = df["KM Rodado"] / (df["Idade"] + 1)
df["Marca"] = df["Título"].str.split().str[0]
display(df[["Descrição", "Motor", "Ano", "Idade", "Km_por_Ano"]].head())

## Separação dos dados antes do pré-processamento

In [ ]:
numericas = ["Idade", "KM Rodado", "Motor", "Km_por_Ano"]
categoricas = ["Marca", "Transmissão", "Estado"]
X = df[numericas + categoricas]
y = df["Preço"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median", keep_empty_features=True), numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),
])
print(f"Treino: {X_train.shape}, Teste: {X_test.shape}")

## Clusterização descritiva, separada da regressão

In [ ]:
X_cluster = df[["Preço", "Ano"]]
if len(X_cluster.drop_duplicates()) < 3:
    print("São necessários três pares distintos de preço e ano para esta análise.")
else:
    X_scaled = StandardScaler().fit_transform(X_cluster)
    df["Cluster"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_scaled)
    cluster_map = df.groupby("Cluster")["Preço"].mean().sort_values().index
    df["Categoria"] = df["Cluster"].map(dict(zip(
        cluster_map, ("Econômico", "Intermediário", "Premium"), strict=True
    )))
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x="Preço", y="Ano", hue="Categoria", palette="viridis")
    plt.title("Segmentação descritiva dos anúncios (K-Means)")
    plt.show()

Treinamento

In [ ]:
baseline = DummyRegressor(strategy="median").fit(X_train, y_train)
rf_model = make_pipeline(
    prep, RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=2)
).fit(X_train, y_train)

Avaliação do Modelo

In [ ]:
y_pred = rf_model.predict(X_test)
y_base = baseline.predict(X_test)
for nome, pred in [("Mediana do treino", y_base), ("Random Forest", y_pred)]:
    print(f"{nome} | MAE R$ {mean_absolute_error(y_test, pred):,.2f} | "
          f"RMSE R$ {root_mean_squared_error(y_test, pred):,.2f} | R² {r2_score(y_test, pred):.4f}")

Gráfico Real vs Previsto

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6, color='green')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2) 
plt.xlabel('Preço Real')
plt.ylabel('Preço Previsto')
plt.title("Preço anunciado observado vs previsto")
plt.show()

Importância de cada Feature

In [ ]:
impact = permutation_importance(
    rf_model, X_test, y_test, scoring="neg_mean_absolute_error", n_repeats=10, random_state=42
)
importances = pd.Series(impact.importances_mean, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x=importances.values, y=importances.index)
plt.title("Importância por permutação no conjunto de teste")
plt.xlabel("Variação média do MAE")
plt.show()